In [1]:
from pathlib import Path
import shutil
import re

In [2]:
ROOT = Path("~/code/selector_examenes").expanduser()
Path.mkdir(Path("examenes_quimica"), exist_ok=True)

In [3]:
def clasificar_examen(file: Path) -> str | None:

    folder = file.parent.name
    start_position = file.name.find(folder) +1
    text_to_scan = file.name[start_position:].lower()

    parts_added = []
    parts = {"extra" : "extraordinaria",
             "ord" : "ordinaria",
             "titular" : "titular",
             "suplente" : "suplente",
             "reserva a" : "reserva A",
             "reserva b" : "reserva B",
             "junio": "junio",
             "sept": "septiembre"}
    
    for k, v in parts.items():
        if k in text_to_scan:
            parts_added.append(v)

    if parts_added:
        return folder + "_" + "_".join(parts_added) 
    else:
        return file.name

Extraer archivos de subcarpetas

In [4]:
for folder in (ROOT / "examenes_quimica").iterdir():
    if folder.is_dir():
        # find pdf files
        pdf_files = [f for f in folder.rglob("*.pdf")]

        for f in pdf_files:

            # get subfolders of pdf files
            subfolder = str(f.relative_to(folder).parent).replace("/", " - ")

            # move them into folder root
            dest = folder/ f"{subfolder} - {f.name}"

            # move the file to folder base
            shutil.move(f, dest)
            
        # remove subfolders
        for dir in folder.iterdir():
                if dir.is_dir():
                    shutil.rmtree(dir)

Borrar archivos de criterios

In [5]:
for file in (ROOT / "examenes_quimica").rglob("*.pdf"):
    new_file = file.name.lower()

    # remove file if it is a criteria file
    if "criterio" in file.name.lower() or "_c_" in file.name.lower():
        file.unlink(missing_ok=True)
        continue

Limpiar nombres de archivo

In [6]:
def clean_filenames(pattern:str|list, replacement:str="", regex:bool=False) -> None:
    files = list((ROOT / "examenes_quimica").rglob("*.pdf"))
    for f in files:
        if not regex:
            # list of replacements
            if isinstance(pattern, list):
                new_filename = f.name.lower()
                for p in pattern:
                    new_filename = new_filename.replace(p, replacement)
            else:
                # only 1 replacement string
                new_filename = f.name.lower().replace(pattern, replacement)
        else:
            new_filename = re.sub(pattern, replacement, f.name.lower())
        
        shutil.move(f, f.parent / Path(new_filename))
    

In [7]:
# 1. quitar prefijos de carpeta/asignatura
clean_filenames(r"^(sel_\d{4}_)?qu[ií]mica - ", regex=True)

In [8]:
# 2. quitar coletillas irrelevantes
clean_filenames(["exámenes química - a y b", "exámenes química a y b",
                 "-a-b-cabe-anda", "-cabe-anda-1", "-examen-andalucía", "examen_andalucía_",
                 "_20_21", "-impreso-corregido", "-impreso", "impreso-", "_rev"])
#clean_filenames("_ex.pdf", ".pdf")

In [9]:
# 3. quitar 'examen' y variantes de 'química'
clean_filenames(r"examen\d?", regex=True)
clean_filenames(["química", "quimica", "qu╓mica", "quínica"])
clean_filenames(r"(?<![a-z])and(?![a-z])", regex=True)

In [10]:
# # 4. códigos internos de 2010 (610, 210...)
# clean_filenames(r" - \d10(?=\.pdf$)", regex=True)

In [11]:
# 5. unificar separadores
clean_filenames([" - ", " ", "-"], "_")
clean_filenames(r"_{2,}", "_", regex=True)
clean_filenames(r"^_+|_+(?=\.pdf$)", regex=True)

In [12]:
# 6. normalizar términos
clean_filenames(r"(?<![a-z])sept(?![a-z])", "septiembre", regex=True)
clean_filenames(r"(?<![a-z])jun(?![a-z])", "junio", regex=True)
clean_filenames(r"(?<![a-z])ord(?![a-z])", "ordinaria", regex=True)
clean_filenames(r"(?<![a-z])extra(?![a-z])", "extraordinaria", regex=True)
clean_filenames(r"(?<![a-z])ra(?![a-z])", "reserva_a", regex=True)
clean_filenames(r"(?<![a-z])rb(?![a-z])", "reserva_b", regex=True)
clean_filenames(r"(?<![a-z])reser(?![a-z])", "reserva", regex=True)
clean_filenames("suplemente", "suplente")

In [13]:
# 7. palabras repetidas (junio_junio, titular_titular)
clean_filenames(r"(?<![a-z])([a-z]+)_\1(?![a-z])", r"\1", regex=True)

In [14]:
# 8. números sobrantes tras titular/reserva/mes
clean_filenames(r"(titular|reserva_[ab]|junio|septiembre)_\d(?=\.pdf$)", r"\1", regex=True)

In [15]:
# 9. orden: convocatoria primero
clean_filenames(r"^(titular|suplente|reserva_[ab])_(junio|septiembre)", r"\2_\1", regex=True)

In [16]:
# 10. detalles finales
clean_filenames(r"_(junio|julio)(?=\.pdf$)", regex=True)
clean_filenames(r"suplente(\d)", r"suplente_\1", regex=True)

In [17]:
# 11. eliminar 3 números al final del nombre de archivo
clean_filenames(r"_\d{3}\.pdf$", ".pdf", regex=True)

¡¡RENOMBRAR EXAMEN 2015 QUÍMICA 1.PDF A 2.PDF!!

In [18]:
# 12. eliminar universidad de almería (ua)
clean_filenames(r"^ua", "", regex=True)

Insertar año y asignatura

In [19]:
for pdf in sorted((ROOT / "examenes_quimica").rglob("*.pdf")):
    year_subject = pdf.parts[-2]
    new_file = year_subject + "_" + pdf.name
    shutil.move(pdf, pdf.parent / f"{new_file}")    

In [20]:
for folder in sorted((ROOT / "examenes_quimica").iterdir()):
    if folder.is_dir():
        files_count = len(list(folder.iterdir()))
        if  files_count != 6:
            print(f"Error en carpeta {folder}. {files_count} archivos.")

Error en carpeta /Users/daniel/code/selector_examenes/examenes_quimica/2012_quimica. 14 archivos.
Error en carpeta /Users/daniel/code/selector_examenes/examenes_quimica/2015_quimica. 5 archivos.
Error en carpeta /Users/daniel/code/selector_examenes/examenes_quimica/2016_quimica. 5 archivos.


- 2012 tiene las opciones por separado (opciones A y B). Además, en dos de los exámenes están las opciones por separado.y un pdf con las dos opciones juntas. Requiere manipulación de los ejercicios.

- 2015: necesario renombrar 1.pdf a 2.pdf antes de aplicar el cambio 12.
  
- 2016 tiene solo 5 ficheros. Falta un exame.